# ThreatLens on IBM LinuxONE: evidence notebook
**Team Stalactite · IBM Z Datathon 2026 · Theme: AI Secured: Innovation Without Exposure**

Every cell below runs inside the `notebook` container on our IBM LinuxONE Community Cloud server (s390x).
It shows the platform, the corpus, the trained model making predictions with masked input, the measured
results, and the local IBM Granite explanation step. No report text leaves the server.

In [1]:
import json, os, platform, sys, time
from importlib import metadata
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
features = next((l for l in open("/proc/cpuinfo") if l.startswith("features")), "")
print("Architecture      :", platform.machine())
print("Python            :", platform.python_version())
print("CPUs / memory     :", os.cpu_count(), "CPUs /", round(int(open('/proc/meminfo').readline().split()[1]) / 1024**2, 1), "GB")
print("NNPA (on-chip AI) :", "present" if "nnpa" in features.split() else "not listed")
for p in ["scikit-learn", "numpy", "pandas", "llama_cpp_python", "ibm-zdnn-plugin", "torch_nnpa"]:
    try: print(f"{p:18}:", metadata.version(p))
    except metadata.PackageNotFoundError: print(f"{p:18}: not installed")

Architecture      : s390x
Python            : 3.12.15
CPUs / memory     : 3 CPUs / 7.8 GB
NNPA (on-chip AI) : present
scikit-learn      : 1.9.0
numpy             : 2.0.2
pandas            : 3.0.6
llama_cpp_python  : 0.3.36
ibm-zdnn-plugin   : 2.3.0
torch_nnpa        : 1.5.0+gite861108


## 1. The corpus (built on LinuxONE by `scripts/build_corpus.py`)

In [2]:
stats = json.loads((ROOT / "data/processed/stats.json").read_text())
for k in ["unique_rows", "rows_per_split", "distinct_techniques_per_split", "rows_with_leakage_or_markup_removed", "eval_clean_rows", "dropped"]:
    print(f"{k:38}", stats[k])

unique_rows                            24872
rows_per_split                         {'train': 15872, 'val': 2883, 'test': 3710, 'external_test': 2407}
distinct_techniques_per_split          {'train': 497, 'val': 362, 'test': 372, 'external_test': 345}
rows_with_leakage_or_markup_removed    {'tram': 71, 'procedure': 28, 'derived_procedure': 213, 'expert': 26, 'annoctr': 667, 'cisa': 244}
eval_clean_rows                        {'val': 2306, 'test': 3028, 'external_test': 2329}
dropped                                {'too short': 194, 'no valid technique label': 184, 'heading or caption': 465, 'too long': 9}


## 2. Load the trained model and analyse a passage (sensitive values masked first)

In [3]:
from threatlens.model import TechniqueRanker
from threatlens.explain import Explainer
from threatlens.redact import redact
t0 = time.perf_counter()
model = TechniqueRanker.load(ROOT / "models/threatlens_final.joblib")
explainer = Explainer(model)
print(f"Model loaded in {time.perf_counter() - t0:.1f}s: {len(model.labels_)} techniques, {model.n_features_:,} features")

passage = ("The attackers used PowerShell to download additional tools from 185.141.61[.]246, "
           "then dumped credentials from LSASS memory. They emailed staff at evil-corp.com with malicious Word attachments.")
masked, counts = redact(passage)
print()
print("Masked passage:", masked)
print("Masked values :", counts)
t0 = time.perf_counter()
sentences, top, alternatives = explainer.suggest(masked, k=3)
print()
print(f"Top 3 techniques on {platform.machine()} in {(time.perf_counter() - t0) * 1000:.0f} ms:")
for i, s in enumerate(top, 1):
    print(f"  {i}. {s['technique_id']:10} score {s['score']:.2f}  evidence words: {', '.join(s['evidence_terms'])}")

Model loaded in 1.2s: 415 techniques, 126,778 features

Masked passage: The attackers used PowerShell to download additional tools from [IP], then dumped credentials from LSASS memory. They emailed staff at [DOMAIN] with malicious Word attachments.
Masked values : {'IP': 1, 'DOMAIN': 1}

Top 3 techniques on s390x in 61 ms:
  1. T1059.001  score 0.75  evidence words: used powershell, download additional
  2. T1003.001  score 0.73  evidence words: credentials, lsass memory
  3. T1105      score 0.37  evidence words: download, additional, tools, attackers


## 3. Measured results (`scripts/train_baseline.py`: tuned on validation, test and CISA scored once)

In [4]:
import pandas as pd
m = json.loads((ROOT / "outputs/metrics/baseline_metrics.json").read_text())
rows = []
for split in ["test", "external_test"]:
    for subset in ["all", "clean"]:
        for level in ["technique", "parent"]:
            r = m["results"][split][subset][level]
            rows.append({"split": "CISA (external)" if split == "external_test" else split, "subset": subset, "level": level,
                         "rows": r["rows"], "hit@1": r["hit@1"], "hit@3": r["hit@3"], "hit@5": r["hit@5"], "micro_f1": r["micro_f1"]})
print(f"Model: {m['model']} | C={m['C']} | {m['labels_in_model']} techniques | fit {m['fit_seconds']}s | {m['latency_ms_per_sentence']} ms/sentence | built on {m['platform']}")
pd.DataFrame(rows)

Model: TF-IDF (word 1-2 + char 3-5 n-grams) + one-vs-rest logistic regression | C=16.0 | 397 techniques | fit 103.9s | 40.3 ms/sentence | built on s390x / Python 3.12.15


,split,subset,level,rows,hit@1,hit@3,hit@5,micro_f1
0,test,all,technique,3710,0.6224,0.8089,0.8582,0.5733
1,test,all,parent,3710,0.7073,0.8655,0.9062,0.6577
2,test,clean,technique,3028,0.6156,0.7972,0.8507,0.5594
3,test,clean,parent,3028,0.6879,0.8534,0.8970,0.6334
4,CISA (external),all,technique,2407,0.4155,0.5916,0.6631,0.3903
5,CISA (external),all,parent,2407,0.4931,0.6751,0.7474,0.4631
6,CISA (external),clean,technique,2329,0.4062,0.5839,0.6565,0.3815
7,CISA (external),clean,parent,2329,0.4835,0.6685,0.7419,0.4533


## 4. Generative step: IBM Granite explains a suggestion, locally (llama.cpp built with IBM zDNN)

In [5]:
from threatlens.granite import Granite
tech = pd.read_csv(ROOT / "data/processed/techniques.csv").set_index("technique_id")
g = Granite()
for s in top[:2]:
    tid = s["technique_id"]
    r = g.explain(s["evidence"], tid, tech.loc[tid, "full_name"], "")
    print(f"{tid} {tech.loc[tid, 'full_name']}")
    print(f"  Granite ({r['model']}, answered in {r['seconds']}s on {platform.machine()}): {r['text']}")

llama_kv_cache: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 256


T1059.001 Command and Scripting Interpreter: PowerShell
  Granite (granite-4.0-h-350m-base-be.f16.gguf, answered in 5.03s on s390x): The report says the attackers used PowerShell to download additional tools from [IP], then dumped credentials from LSASS memory.


T1003.001 OS Credential Dumping: LSASS Memory
  Granite (granite-4.0-h-350m-base-be.f16.gguf, answered in 4.18s on s390x): The report says the attackers dumped credentials from LSASS memory, which is credential dumping.


## 5. Examples from CISA advisories the model never saw in training

In [6]:
ex = pd.read_csv(ROOT / "outputs/metrics/baseline_examples.csv")
ex = ex[(ex.eval_clean == "yes") & (ex.text.str.len() < 200)]
pd.set_option("display.max_colwidth", 140)
pd.concat([ex[ex.hit_in_top3].head(4), ex[~ex.hit_in_top3].head(3)])[["hit_in_top3", "gold", "top3", "text"]]

,hit_in_top3,gold,top3,text
0,True,T1078,T1078;T1486;T1078.002,Play ransomware actors obtain and abuse existing account credentials to gain initial access.
1,True,T1190,T1190;T1486;T1068,Play ransomware actors exploit vulnerabilities in internet-facing systems to gain access to networks.
2,True,T1133,T1486;T1133;T1078,"Play ransomware actors have used remote access services, such as RDP/VPN connection to gain initial access."
3,True,T1059.001,T1059.001;T1486;T1190,Play ransomware actors leveraged PowerShell commands to achieve RCE with a newly disclosed vulnerability.
7,False,T1685.005,T1486;T1070.004;T1036.005,Play ransomware actors delete logs or other indicators of compromise to hide intrusion activity.
8,False,T1552,T1486;T1078;T1570,Play ransomware actors attempt to identify and exploit credentials stored insecurely on a compromised network.
10,False,T1570,T1486;T1105;T1685,Play ransomware actors distribute executables within the compromised environment.
